# Tools Smoke Test

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))

In [ ]:
import json

import sys
import os
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

# Ensure repo root is on path when running from notebooks/
repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Set DOCS_DIR relative to repo root if not already set
if not os.environ.get("DOCS_DIR"):
    os.environ["DOCS_DIR"] = str(repo_root / "docs")

## BigQuery Tools

In [ ]:
from tools.bigquery_tools import get_latest_transactions

In [ ]:
txns = get_latest_transactions("CUST-001")
print(json.dumps(txns, indent = 2))

## GCS Tools

In [ ]:
from tools.gcs_tools import get_document_catalog, get_document

In [ ]:
catalog = list(get_document_catalog())
print(json.dumps(catalog, indent = 2))

In [ ]:
first_doc = get_document_catalog()[0]
document_id = first_doc["document_id"]
content = get_document(document_id)
print(content[:500])

In [ ]:
from tools.gcs_tools import get_document_metadata

meta = get_document_metadata(document_id)
print(meta)

In [ ]:
meta_with_content = get_document_metadata(document_id, include_content=True)
print("keys:", list(meta_with_content.keys()))
print("\ncontent (first 200 chars):")
print(meta_with_content["content"][:200])

## LLM Tools

In [ ]:
from tools.llm_tools import get_embeddings

In [ ]:
text = content[:200]
embedding = get_embeddings().embed_query(text)
print(f"Vector dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")


In [ ]:
from tools.llm_tools import get_llm
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
response = get_llm().invoke([
    SystemMessage(content = 'Respondeme en castellano'),
    HumanMessage(content = 'Respondé en una oración: ¿qué es el lavado de activos?'),
]).content
print(response)

## PostgreSQL Tools

In [ ]:
from tools.postgresql_tools import ensure_table, search_similar

In [ ]:
ensure_table()
print("Table ready")

In [ ]:
results = search_similar(embedding, top_k=3, country_code="CO")
print(results)

## Elasticsearch Tools

In [ ]:
from tools.elasticsearch_tools import ensure_index, search_documents

In [ ]:
ensure_index()
print("Index ready")

In [ ]:
results = search_documents("transferencia internacional persona jurídica", top_k=3, country_code="CO")
print(results)

## FalkorDB Tools

In [ ]:
from tools.falkordb_tools import create_document_nodes, create_edge, get_related_documents

In [ ]:
docs = [
    {"document_id": "smoke-doc-a", "country_code": "CO"},
    {"document_id": "smoke-doc-b", "country_code": "CO"},
]
create_document_nodes(docs)
create_edge("smoke-doc-a", "smoke-doc-b", expansion="forward")
print("Nodes and edge created")

In [ ]:
related = get_related_documents("smoke-doc-a")
print(related)  # expected: ['smoke-doc-b']

## Clear All Stores

In [ ]:
from tools.postgresql_tools import clear_table
from tools.elasticsearch_tools import clear_index
from tools.falkordb_tools import clear_graph

clear_table()
clear_index()
clear_graph()
print("All stores cleared")